In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
!pip install groq --quiet
print("✅ Groq instalado.")

In [ ]:
from google.colab import userdata
FINNO_API_KEY = userdata.get('FINNO_GROQ_TOKEN')
print("✅ Chave carregada.")

In [ ]:
import json
import csv
from pathlib import Path

def carregar_perfil(caminho="data/perfil_investidor.json"):
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)

def carregar_produtos(caminho="data/produtos_financeiros.json"):
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)

def carregar_transacoes(caminho="data/transacoes.csv"):
    transacoes = []
    with open(caminho, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row['valor'] = float(row['valor'])
            transacoes.append(row)
    return transacoes

def carregar_historico(caminho="data/historico_atendimento.csv"):
    historico = []
    with open(caminho, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            historico.append(row)
    return historico

def resumir_gastos(transacoes):
    gastos = {}
    for t in transacoes:
        if t['tipo'] == 'debito' and t['categoria'] != 'investimento':
            cat = t['categoria']
            gastos[cat] = gastos.get(cat, 0) + abs(t['valor'])
    return gastos

def filtrar_produtos_por_perfil(produtos_json, perfil_risco):
    return [p for p in produtos_json['produtos'] if perfil_risco in p['indicado_para']]

def formatar_produtos(produtos):
    linhas = []
    for p in produtos:
        linhas.append(
            f"- {p['nome']} ({p['tipo']}): {p['rentabilidade']} | "
            f"Liquidez: {p['liquidez']} | Mínimo: R$ {p['investimento_minimo']:.2f} | "
            f"Risco: {p['risco']}"
        )
    return "\n".join(linhas)

def formatar_historico(historico):
    linhas = []
    for h in historico[-3:]:
        linhas.append(f"- {h['data']} ({h['canal']}): {h['assunto']} → {h['resolucao']}")
    return "\n".join(linhas)

def formatar_metas(perfil):
    linhas = []
    for m in perfil['cliente']['metas'] if 'metas' in perfil['cliente'] else perfil.get('metas', []):
        valor_alvo = m['valor_alvo']
        valor_atual = m['valor_atual']
        progresso = (valor_atual / valor_alvo) * 100
        linhas.append(f"- {m['descricao']}: R$ {valor_atual:,.2f} de R$ {valor_alvo:,.2f} ({progresso:.0f}%)")
    return "\n".join(linhas)


def construir_system_prompt(perfil, produtos_json, transacoes, historico):
    cliente = perfil['cliente']
    perfil_risco = cliente['perfil_risco']

    gastos = resumir_gastos(transacoes)
    gastos_fmt = "\n".join([f"- {cat.capitalize()}: R$ {val:.2f}" for cat, val in gastos.items()])

    produtos_adequados = filtrar_produtos_por_perfil(produtos_json, perfil_risco)
    produtos_fmt = formatar_produtos(produtos_adequados)

    historico_fmt = formatar_historico(historico)
    metas_fmt = formatar_metas(perfil)

    return f"""Você é o Finno, assistente financeiro inteligente do banco digital.
Seu papel é ajudar o cliente a entender suas finanças, tomar decisões de
investimento adequadas ao seu perfil e esclarecer dúvidas sobre produtos.

PERFIL DO CLIENTE:
- Nome: {cliente['nome']}
- Idade: {cliente['idade']} anos
- Renda mensal: R$ {cliente['renda_mensal']:,.2f}
- Patrimônio total: R$ {cliente['patrimonio_total']:,.2f}
- Perfil de risco: {perfil_risco}
- Objetivo principal: {cliente['objetivo_principal']}
- Horizonte de investimento: {cliente['horizonte_investimento_anos']} anos

METAS DO CLIENTE:
{metas_fmt}

RESUMO DE GASTOS DO MÊS ATUAL:
{gastos_fmt}

PRODUTOS DISPONÍVEIS PARA O PERFIL {perfil_risco.upper()}:
{produtos_fmt}

HISTÓRICO RECENTE DE ATENDIMENTO:
{historico_fmt}

REGRAS OBRIGATÓRIAS:
1. Responda sempre em português, de forma clara e objetiva
2. Nunca garanta rentabilidade futura — use "histórico" ou "estimativa"
3. Para decisões acima de R$ 10.000, sugira validar com assessor humano
4. Recuse educadamente perguntas fora do escopo financeiro
5. Use apenas os dados fornecidos acima — não invente taxas ou produtos
6. Quando não tiver informação disponível, diga claramente
"""

from groq import Groq

def iniciar_finno(api_key):
    perfil = carregar_perfil()
    produtos = carregar_produtos()
    transacoes = carregar_transacoes()
    historico = carregar_historico()

    system_prompt = construir_system_prompt(perfil, produtos, transacoes, historico)

    client = Groq(api_key=api_key)
    historico_conversa = []

    print("=" * 55)
    print("  Finno — Assistente Financeiro Inteligente")
    print("=" * 55)
    print(f"  Olá, {perfil['cliente']['nome'].split()[0]}! Como posso ajudar?")
    print("  (digite 'sair' para encerrar)\n")

    while True:
        pergunta = input("Você: ").strip()

        if pergunta.lower() in ['sair', 'exit', 'quit']:
            print("\nFinno: Até logo! Qualquer dúvida financeira, estarei aqui.")
            break

        if not pergunta:
            continue

        historico_conversa.append({"role": "user", "content": pergunta})

        resposta = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": system_prompt},
                *historico_conversa
            ],
            max_tokens=500,
            temperature=0.4
        )

        texto = resposta.choices[0].message.content
        historico_conversa.append({"role": "assistant", "content": texto})

        print(f"\nFinno: {texto}\n")
        print("-" * 55)

FINNO_API_KEY = "gsk_xxxxxxxxxx"

In [ ]:
import json
import csv

# Sample data for perfil_investidor.json
perfil_investidor_data = {
    "cliente": {
        "nome": "Mariana Silva",
        "idade": 30,
        "renda_mensal": 5000.00,
        "patrimonio_total": 75000.00,
        "perfil_risco": "moderado",
        "objetivo_principal": "crescimento_patrimonial",
        "horizonte_investimento_anos": 5,
        "metas": [
            {
                "descricao": "Comprar apartamento",
                "valor_alvo": 250000.00,
                "valor_atual": 50000.00
            },
            {
                "descricao": "Reserva de emergência",
                "valor_alvo": 15000.00,
                "valor_atual": 10000.00
            }
        ]
    }
}

# Create perfil_investidor.json
with open("data/perfil_investidor.json", "w", encoding="utf-8") as f:
    json.dump(perfil_investidor_data, f, indent=4)
print("✅ Arquivo 'data/perfil_investidor.json' criado.")

# Sample data for produtos_financeiros.json
produtos_financeiros_data = {
    "produtos": [
        {
            "nome": "CDB Moderado",
            "tipo": "renda_fixa",
            "rentabilidade": "105% CDI",
            "liquidez": "D+1",
            "investimento_minimo": 1000.00,
            "risco": "baixo",
            "indicado_para": ["conservador", "moderado"]
        },
        {
            "nome": "Fundo Multimercado Balanceado",
            "tipo": "fundos_investimento",
            "rentabilidade": "CDI + 2% (histórico)",
            "liquidez": "D+30",
            "investimento_minimo": 5000.00,
            "risco": "moderado",
            "indicado_para": ["moderado", "arrojado"]
        },
        {
            "nome": "Ações Blue Chips",
            "tipo": "renda_variavel",
            "rentabilidade": "Variável",
            "liquidez": "D+2",
            "investimento_minimo": 500.00,
            "risco": "alto",
            "indicado_para": ["arrojado"]
        }
    ]
}

# Create produtos_financeiros.json
with open("data/produtos_financeiros.json", "w", encoding="utf-8") as f:
    json.dump(produtos_financeiros_data, f, indent=4)
print("✅ Arquivo 'data/produtos_financeiros.json' criado.")

In [ ]:
# Sample data for transacoes.csv
transacoes_data = [
    {"data": "2024-03-01", "descricao": "Salário", "tipo": "credito", "valor": 5000.00, "categoria": "renda"},
    {"data": "2024-03-05", "descricao": "Aluguel", "tipo": "debito", "valor": -1500.00, "categoria": "moradia"},
    {"data": "2024-03-08", "descricao": "Supermercado", "tipo": "debito", "valor": -400.00, "categoria": "alimentacao"},
    {"data": "2024-03-10", "descricao": "Lazer", "tipo": "debito", "valor": -200.00, "categoria": "lazer"},
    {"data": "2024-03-15", "descricao": "Investimento CDB", "tipo": "debito", "valor": -1000.00, "categoria": "investimento"}
]

# Create transacoes.csv
with open("data/transacoes.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=transacoes_data[0].keys())
    writer.writeheader()
    writer.writerows(transacoes_data)
print("✅ Arquivo 'data/transacoes.csv' criado.")

In [ ]:
# Sample data for historico_atendimento.csv
historico_atendimento_data = [
    {"data": "2024-01-20", "canal": "chat", "assunto": "Dúvida sobre CDB", "resolucao": "Explicado as características do CDB."},
    {"data": "2024-02-10", "canal": "telefone", "assunto": "Aporte em fundo multimercado", "resolucao": "Auxiliado no processo de aporte."},
    {"data": "2024-03-03", "canal": "chat", "assunto": "Consulta de saldo", "resolucao": "Informado o saldo atualizado."}
]

# Create historico_atendimento.csv
with open("data/historico_atendimento.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=historico_atendimento_data[0].keys())
    writer.writeheader()
    writer.writerows(historico_atendimento_data)
print("✅ Arquivo 'data/historico_atendimento.csv' criado.")

In [ ]:
iniciar_finno(FINNO_API_KEY)